## Backtesting - Points XGBoost model

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("could not find repo root (pyproject.toml)")


ROOT = repo_root()
for folder in (ROOT, ROOT / "notebooks/nba/points/src"):
    folder_text = str(folder)
    if folder_text not in sys.path:
        sys.path.insert(0, folder_text)

from models.shared.artifacts import load_model_bundle
from models.shared.points_given_minutes import nb_logpmf
from models.shared.points_nb import fit_nb2_alpha, over_probability
from points_xgb.data import load_regular_season

MODEL_PATH = ROOT / "models/saved_models/pts_nba_model_production.joblib"
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"missing production model: {MODEL_PATH}")
bundle = load_model_bundle(MODEL_PATH)
models_ho = bundle["quantile_models"]
PTS_FEATURES = list(bundle["feature_names"])
print(
    f"Model {MODEL_PATH.name}: "
    f"trained through {pd.Timestamp(bundle['train_end']).date()}, "
    f"holdout through {pd.Timestamp(bundle['val_end']).date()}"
)
print(f"Features ({len(PTS_FEATURES)}): {', '.join(PTS_FEATURES)}")

# Same dispersion fit as the training notebook. Walk-forward rows choose
# alpha. The holdout is not used.
predictions = pd.read_parquet(ROOT / "data/oos/nba/pts_predictions.parquet")
mean_column = "cal_q_0.50" if "cal_q_0.50" in predictions.columns else "q_0.50"
scored = predictions.loc[np.isfinite(predictions[mean_column])].copy()
n_low = int((scored[mean_column] <= 0).sum())
scored["mu"] = scored[mean_column].clip(lower=0.05)
if n_low:
    print(f"Clipped {n_low:,} non-positive {mean_column} values to 0.05")
oof = scored.loc[~scored["is_holdout"].astype(bool)]
alpha = fit_nb2_alpha(oof["pts"], oof["mu"])
print(f"NB2 alpha fit on {len(oof):,} walk-forward rows: {alpha:.4f}")


Loaded /Users/alexgonzalez/Documents/nba_quant/models/saved_models/pts_nba_model_2026-10-30.joblib
Model pts_nba_model_2026-10-30.joblib: trained through 2026-04-12, holdout through 2026-10-30
Features (24): fga_rank_10, starting, fta_rolling_20, ts_x_usg, start_rate_10, fg3a_rolling_20, min_lag_1, cfga_rolling_10, cfga_rolling_5, minutes_rolling_10, ufga_rolling_5, opp_def_rating_mean_10, starter_rank, usg_rolling_10, team_games_prev_5d, fg3a_rolling_5, expected_pace, cfga_mean_3_minus_10, tchs_rolling_5, player_team_spread, min_sd_10, implied_team_total, usg_rank_10, tchs_max_10
Clipped 245 non-positive cal_q_0.50 values to 0.05
NB2 alpha fit on 105,235 walk-forward rows: 0.3742


In [10]:
from src.features.points.closing import load_closing_points, match_player_ids
from src.live.predict_points import (
    FEATURES,
    build_points_live,
    lineups_on,
    pinnacle_games_on,
)
from src.pipeline.silver.rotowire import normalize_team_abbreviation

if list(FEATURES) != list(PTS_FEATURES):
    raise AssertionError("live feature list does not match PTS_FEATURES")

# Only these Pacific slate dates are featured / scored.
SLATE_DATES = ("2026-10-29","2026-10-30","2026-10-31")

raw = load_regular_season(ROOT)
lines = load_closing_points(ROOT / "data/props/prizepicks/nba/closing")
lines = match_player_ids(lines, raw)
lines["slate_date"] = pd.to_datetime(lines["slate_date"]).dt.normalize()
wanted = {pd.Timestamp(day).normalize() for day in SLATE_DATES}
lines = lines.loc[lines["slate_date"].isin(wanted)].copy()
if lines.empty:
    raise ValueError(f"No PrizePicks closing lines for {sorted(SLATE_DATES)}")

unmatched = sorted(lines.loc[lines["player_id"].isna(), "player_name"].unique())
print(f"Standard points lines on {', '.join(SLATE_DATES)}: {len(lines):,}")
print(
    f"Unmatched names ({len(unmatched)}): "
    f"{', '.join(unmatched) if unmatched else '(none)'}"
)

name_by_id = (
    raw.sort_values(["game_date", "game_id"], kind="mergesort")
    .groupby("player_id", sort=False)["player_name"]
    .last()
)
raw_dates = pd.to_datetime(raw["game_date"]).dt.normalize()
parts = []
modeled_parts = []
for slate, group in lines.dropna(subset=["player_id"]).groupby("slate_date", sort=True):
    slate = pd.Timestamp(slate).normalize()
    day = slate.date().isoformat()
    before = raw_dates < slate
    same_day = raw_dates.eq(slate)
    print(
        f"{day}: history rows {int(before.sum()):,}, "
        f"{int(same_day.sum()):,} same-day log rows held out of features"
    )
    history = raw.loc[before]
    games = pinnacle_games_on(day)
    lineups = lineups_on(day)
    if not games:
        print(f"  no Pinnacle team file for {day}; spread and total stay missing")
    if not lineups:
        print(f"  no RotoWire file for {day}; starter flags stay missing")
    print(f"  pinnacle games {len(games):,}, rotowire teams {len(lineups):,}")

    player_ids = group["player_id"].drop_duplicates().tolist()
    names = []
    missing_history = []
    id_by_name = {}
    for player_id in player_ids:
        name = name_by_id.get(player_id)
        if name is None or not history["player_name"].eq(name).any():
            missing_history.append(str(name or player_id))
            continue
        names.append(name)
        id_by_name[name] = player_id
    if missing_history:
        print(f"  no games before {day}: {', '.join(missing_history)}")
    if not names:
        raise ValueError(f"no feature rows for {day}")

    # One batched live build per slate; starting from RotoWire posted fives.
    live = build_points_live(
        history,
        names,
        day,
        games=games,
        lineups=lineups,
        extras=(),
    )
    rows = []
    for name, row in live.iterrows():
        player_id = id_by_name[name]
        prior = history.loc[history["player_name"].eq(name)].sort_values(
            ["game_date", "game_id"], kind="mergesort"
        )
        rows.append(
            {
                "player_id": player_id,
                "player_name": name,
                "team_abbreviation": normalize_team_abbreviation(
                    prior["team_abbreviation"].iloc[-1]
                ),
                "history_through": pd.to_datetime(prior["game_date"].iloc[-1]),
                **row.to_dict(),
            }
        )
    featured = pd.DataFrame(rows)
    if featured.empty:
        raise ValueError(f"no feature rows for {day}")

    if not lineups:
        missing_five = featured.index
    else:
        missing_five = featured.index[
            ~featured["team_abbreviation"].isin(lineups)
        ]
    if len(missing_five):
        featured.loc[missing_five, "starting"] = np.nan
        print(f"  {len(missing_five):,} players had no posted five")

    through = pd.to_datetime(featured["history_through"]).dt.normalize()
    if not (through < slate).all():
        raise AssertionError(f"{day} used a game on or after the closing date")
    print(
        f"  history through {through.max().date()}, "
        f"{len(featured):,} players featured"
    )
    played = (
        raw.loc[same_day, ["player_id", "pts"]]
        .drop_duplicates("player_id")
        .rename(columns={"pts": "pts_played"})
    )
    overlap = [
        column
        for column in featured.columns
        if column in group.columns and column != "player_id"
    ]
    board = featured.loc[:, ["player_id", "history_through", *PTS_FEATURES]].copy()
    board["slate_date"] = pd.Timestamp(slate).normalize()
    board = board.merge(played, on="player_id", how="left")
    modeled_parts.append(board)
    parts.append(
        group.merge(featured.drop(columns=overlap), on="player_id", how="left").merge(
            played, on="player_id", how="left"
        )
    )

closing = pd.concat(parts, ignore_index=True)
missing = [column for column in PTS_FEATURES if column not in closing.columns]
if missing:
    raise KeyError(f"Closing frame is missing features: {missing}")

closing[PTS_FEATURES] = closing[PTS_FEATURES].apply(pd.to_numeric, errors="coerce")
ready = closing["min_lag_1"].notna().to_numpy()
mu = np.full(len(closing), np.nan)
p_over = np.full(len(closing), np.nan)
if ready.any():
    predicted = np.clip(
        models_ho["q_0.50"].predict(closing.loc[ready, PTS_FEATURES]),
        0.05,
        None,
    )
    mu[ready] = predicted
    p_over[ready] = over_probability(predicted, alpha, closing.loc[ready, "line"])
unscored = int((~ready).sum())
if unscored:
    print(f"{unscored:,} lines had no prior games and were not scored")
closing["mu"] = mu
closing["p_over"] = p_over
closing["went_over"] = closing["pts_played"].gt(closing["line"])
modeled = pd.concat(modeled_parts, ignore_index=True)
modeled[PTS_FEATURES] = modeled[PTS_FEATURES].apply(pd.to_numeric, errors="coerce")
modeled_ready = modeled["min_lag_1"].notna()
modeled["mu"] = np.nan
if modeled_ready.any():
    modeled.loc[modeled_ready, "mu"] = np.clip(
        models_ho["q_0.50"].predict(modeled.loc[modeled_ready, PTS_FEATURES]),
        0.05,
        None,
    )
print(
    f"Priced {closing['p_over'].notna().sum():,} PrizePicks lines on "
    f"{', '.join(SLATE_DATES)}. "
    f"Model means saved for {len(modeled):,} player-slates."
)
closing[
    [
        "slate_date",
        "player_name",
        "line",
        "mu",
        "p_over",
        "pts_played",
        "history_through",
    ]
].sort_values(["slate_date", "p_over"], ascending=[True, False]).head(20)


Standard points lines on 2026-10-29, 2026-10-30, 2026-10-31: 270
Unmatched names (0): (none)
2026-10-29: history rows 178,269, 66 same-day log rows held out of features
  pinnacle games 177, rotowire teams 6


/Users/alexgonzalez/Documents/nba_quant/src/features/points/scoring.py:196: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f"{name}_max_10"] = prior_roll(
/Users/alexgonzalez/Documents/nba_quant/src/features/points/scoring.py:189: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f"{name}_min_10"] = prior_roll(
/Users/alexgonzalez/Documents/nba_quant/src/features/points/scoring.py:196: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor per

  2 players had no posted five
  history through 2026-10-28, 47 players featured
2026-10-30: history rows 178,335, 232 same-day log rows held out of features
  pinnacle games 105, rotowire teams 20


/Users/alexgonzalez/Documents/nba_quant/src/features/points/scoring.py:196: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f"{name}_max_10"] = prior_roll(
/Users/alexgonzalez/Documents/nba_quant/src/features/points/scoring.py:189: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f"{name}_min_10"] = prior_roll(
/Users/alexgonzalez/Documents/nba_quant/src/features/points/scoring.py:196: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor per

  history through 2026-10-28, 122 players featured
2026-10-31: history rows 178,567, 161 same-day log rows held out of features
  pinnacle games 143, rotowire teams 14


/Users/alexgonzalez/Documents/nba_quant/src/features/points/scoring.py:196: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f"{name}_max_10"] = prior_roll(
/Users/alexgonzalez/Documents/nba_quant/src/features/points/scoring.py:189: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f"{name}_min_10"] = prior_roll(
/Users/alexgonzalez/Documents/nba_quant/src/features/points/scoring.py:196: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor per

  history through 2026-10-30, 101 players featured
Priced 270 PrizePicks lines on 2026-10-29, 2026-10-30, 2026-10-31. Model means saved for 270 player-slates.


,slate_date,player_name,line,mu,p_over,pts_played,history_through
15,2026-10-29,Dyson Daniels,10.5,13.230927,0.547579,4.0,2026-10-28
18,2026-10-29,Isaiah Stewart,5.5,6.893628,0.534214,18.0,2026-10-26
26,2026-10-29,Jeremiah Fears,15.5,18.573830,0.523354,22.0,2026-10-28
44,2026-10-29,Zach Edey,8.0,10.208859,0.518682,13.0,2026-10-26
36,2026-10-29,Precious Achiuwa,10.5,12.441268,0.513594,24.0,2026-10-27
21,2026-10-29,James Harden,17.5,20.279119,0.504867,24.0,2026-10-28
35,2026-10-29,Peyton Watson,13.5,15.582422,0.500940,4.0,2026-10-28
33,2026-10-29,Nickeil Alexander-Walker,18.5,21.146219,0.497192,25.0,2026-10-28
29,2026-10-29,Karlo Matković,7.5,8.640153,0.494490,12.0,2026-10-28
34,2026-10-29,Onyeka Okongwu,11.5,13.114491,0.492888,29.0,2026-10-28


In [8]:
import json
from pathlib import Path

from src.features.points.closing import match_player_ids

POINTS_STATS = {"points", "pts"}
BOOK_CLOSING = {
    "prizepicks": ROOT / "data/props/prizepicks/nba/closing",
    "underdog": ROOT / "data/props/underdogs/nba/closing",
    "novig": ROOT / "data/props/novig/nba/closing",
    "prophetx": ROOT / "data/props/prophetx/nba/closing",
    "pinnacle": ROOT / "data/props/pinnacle/nba/closing",
}


def _american_profit(price):
    """Profit on a $1 stake. Negative American odds risk more than they pay."""
    price = np.asarray(price, dtype=float)
    return np.where(price < 0, 100.0 / np.abs(price), price / 100.0)


def _implied(price):
    price = price.astype(float)
    negative = price < 0
    out = pd.Series(np.nan, index=price.index, dtype=float)
    out.loc[negative] = price.loc[negative].abs() / (price.loc[negative].abs() + 100)
    out.loc[~negative] = 100 / (price.loc[~negative] + 100)
    return out


def _load_points_sides(book, directory):
    rows = []
    folder = Path(directory)
    for path in sorted(folder.glob("*_nba_closing_*.json")):
        payload = json.loads(path.read_text())
        slate = pd.Timestamp(payload["date"]).normalize()
        for player in payload.get("players", []):
            for offer in player.get("lines", []):
                stat = str(offer.get("stat_type") or "").strip().lower()
                if stat not in POINTS_STATS:
                    continue
                if book == "prizepicks" and offer.get("odds_type") != "standard":
                    continue
                rows.append(
                    {
                        "slate_date": slate,
                        "book": book,
                        "player_name": player.get("player"),
                        "line": float(offer["line_score"]),
                        "side": offer.get("side"),
                        "american_price": offer.get("american_price"),
                    }
                )
    return pd.DataFrame(rows)


def _main_points_line(sides):
    """One two-way points line per player.

    Novig and ProphetX keep alternate ladders. The main line is the
    rung where over and under are closest to each other. PrizePicks
    standard is one line, priced -119 on both sides.
    """
    prizepicks = sides.loc[sides["book"].eq("prizepicks")].copy()
    prizepicks = prizepicks.drop_duplicates(["slate_date", "player_name", "line"])
    prizepicks["over_american"] = -119
    prizepicks["under_american"] = -119

    other = sides.loc[~sides["book"].eq("prizepicks")].dropna(
        subset=["side", "american_price"]
    )
    other = other.assign(side=other["side"].astype("string").str.lower())
    if other.empty:
        wide = pd.DataFrame(
            columns=[
                "slate_date",
                "book",
                "player_name",
                "line",
                "over",
                "under",
            ]
        )
        one_sided = 0
    else:
        wide = (
            other.pivot_table(
                index=["slate_date", "book", "player_name", "line"],
                columns="side",
                values="american_price",
                aggfunc="first",
            )
            .reset_index()
        )
        for column in ("over", "under"):
            if column not in wide.columns:
                wide[column] = np.nan
        paired = wide["over"].notna() & wide["under"].notna()
        one_sided = int((~paired).sum())
        wide = wide.loc[paired].copy()
        wide["gap"] = (_implied(wide["over"]) - _implied(wide["under"])).abs()
        wide = wide.sort_values("gap").drop_duplicates(
            ["slate_date", "book", "player_name"]
        )
    wide = wide.rename(columns={"over": "over_american", "under": "under_american"})
    keep = [
        "slate_date",
        "book",
        "player_name",
        "line",
        "over_american",
        "under_american",
    ]
    markets = pd.concat([prizepicks[keep], wide[keep]], ignore_index=True)
    return markets, one_sided


sides = pd.concat(
    [_load_points_sides(book, folder) for book, folder in BOOK_CLOSING.items()],
    ignore_index=True,
)
sides["slate_date"] = pd.to_datetime(sides["slate_date"]).dt.normalize()
# Reuse SLATE_DATES from the feature cell (defaults to 2026-10-30).
_slate_dates = globals().get("SLATE_DATES", ("2026-10-30",))
_wanted = {pd.Timestamp(day).normalize() for day in _slate_dates}
sides = sides.loc[sides["slate_date"].isin(_wanted)].copy()
if sides.empty:
    raise ValueError(f"No closing sides for {_slate_dates}")
print(
    f"Loaded {len(sides):,} point sides on "
    f"{', '.join(str(pd.Timestamp(d).date()) for d in sorted(_wanted))} "
    f"across {sides['book'].nunique()} books"
)
markets, one_sided = _main_points_line(sides)
markets = match_player_ids(markets, raw)
unmatched = sorted(markets.loc[markets["player_id"].isna(), "player_name"].unique())

means = modeled if "modeled" in globals() else closing
means = means.copy()
if "slate_date" not in means.columns and "slate_date_x" in means.columns:
    means = means.rename(
        columns={"slate_date_x": "slate_date", "player_name_x": "player_name"}
    )
means["slate_date"] = pd.to_datetime(means["slate_date"]).dt.normalize()
means = means.loc[:, ["slate_date", "player_id", "mu", "pts_played"]].drop_duplicates(
    ["slate_date", "player_id"]
)
book = markets.merge(means, on=["slate_date", "player_id"], how="left")
missing_mu = int(book["mu"].isna().sum())
book = book.dropna(subset=["mu"]).reset_index(drop=True)

level = book["line"].to_numpy()
mean = book["mu"].to_numpy()
p_over = over_probability(mean, alpha, level)
integer = np.isclose(level, np.rint(level))
p_push = np.zeros(len(book))
if integer.any():
    p_push[integer] = np.exp(nb_logpmf(level[integer], mean[integer], alpha))
p_under = 1.0 - p_over - p_push
over_profit = _american_profit(book["over_american"])
under_profit = _american_profit(book["under_american"])
book["p_over"] = p_over
book["p_under"] = p_under
book["ev_over"] = p_over * over_profit - p_under
book["ev_under"] = p_under * under_profit - p_over
take_over = book["ev_over"] >= book["ev_under"]
book["side"] = np.where(take_over, "over", "under")
book["p_side"] = np.where(take_over, p_over, p_under)
book["price"] = np.where(take_over, book["over_american"], book["under_american"])
book["ev"] = np.maximum(book["ev_over"], book["ev_under"])
book["win_profit"] = np.where(take_over, over_profit, under_profit)

settled = book["pts_played"].notna()
won = np.where(
    take_over,
    book["pts_played"].gt(book["line"]),
    book["pts_played"].lt(book["line"]),
)
pushed = book["pts_played"].eq(book["line"])
book["result"] = np.where(
    ~settled,
    "pending",
    np.where(pushed, "push", np.where(won, "win", "loss")),
)
book["profit"] = np.where(
    book["result"].eq("win"),
    book["win_profit"],
    np.where(book["result"].eq("loss"), -1.0, 0.0),
)

graded = book.loc[book["result"].isin(["win", "loss"])].copy()
plus = book.loc[book["ev"].gt(0)].copy()
plus_graded = plus.loc[plus["result"].isin(["win", "loss"])]
print(
    f"Main points closes: {len(markets):,}. "
    f"One-sided offers left out: {one_sided:,}. "
    f"Unmatched names ({len(unmatched)}): "
    f"{', '.join(unmatched) if unmatched else '(none)'}"
)
print(
    f"Scored {len(book):,} lines with a model mean. "
    f"{missing_mu:,} had no mean. Alpha {alpha:.4f}."
)
print(
    f"Settled {int(settled.sum()):,}. "
    f"Pushes {int((book['result'] == 'push').sum()):,}. "
    f"+EV {len(plus):,} ({int(plus['result'].ne('pending').sum()):,} settled)."
)
if len(graded):
    print(
        f"Favored side hit {graded['result'].eq('win').mean():.1%} "
        f"on {len(graded):,} decided lines "
        f"(mean probability {graded['p_side'].mean():.1%})."
    )
if len(plus_graded):
    plus_settled = plus.loc[plus["result"].ne("pending")]
    print(
        f"+EV hit {plus_graded['result'].eq('win').mean():.1%} "
        f"on {len(plus_graded):,} decided bets. "
        f"ROI {plus_settled['profit'].sum() / len(plus_settled):.1%} per $1 stake."
    )

by_book = (
    book.loc[book["result"].ne("pending")]
    .groupby("book", observed=False)
    .apply(
        lambda frame: pd.Series(
            {
                "n": len(frame),
                "decided": int(frame["result"].isin(["win", "loss"]).sum()),
                "hit_rate": frame.loc[
                    frame["result"].isin(["win", "loss"]), "result"
                ]
                .eq("win")
                .mean(),
                "roi": frame["profit"].sum() / len(frame),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)
print("Settled closes by book. ROI is profit per $1 at that book's price.")
display(by_book)

# Calibration: when the model says X%, do those bets hit about X%?
prob_edges = [0.50, 0.55, 0.60, 0.65, 0.70, 1.01]
prob_labels = ["50\u201355%", "55\u201360%", "60\u201365%", "65\u201370%", "70%+"]
graded["bucket"] = pd.cut(
    graded["p_side"],
    bins=prob_edges,
    labels=prob_labels,
    right=False,
    include_lowest=True,
)
accuracy = (
    graded.groupby("bucket", observed=False)
    .agg(
        n=("result", "size"),
        wins=("result", lambda series: int(series.eq("win").sum())),
        actual_hit_rate=("result", lambda series: series.eq("win").mean()),
        predicted_prob=("p_side", "mean"),
        roi=("profit", "mean"),
    )
    .reset_index()
)
accuracy["calibration_gap"] = (
    accuracy["actual_hit_rate"] - accuracy["predicted_prob"]
)
print(
    "Favored-side calibration by predicted win probability. "
    "predicted_prob is the bucket's mean model probability; "
    "actual_hit_rate is how often that side won; "
    "calibration_gap = actual - predicted (near 0 is well calibrated)."
)
display(accuracy.round(3))

# Same check on raw P(over), not just the favored side.
over_decided = book.loc[
    book["pts_played"].notna() & book["p_over"].notna()
].copy()
over_decided["over_hit"] = over_decided["pts_played"].gt(over_decided["line"])
# Push rows (exact line) are neither over hits nor misses for this table.
over_decided = over_decided.loc[
    ~over_decided["pts_played"].eq(over_decided["line"])
].copy()
over_edges = [0.0, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 1.01]
over_labels = [
    "<40%",
    "40\u201345%",
    "45\u201350%",
    "50\u201355%",
    "55\u201360%",
    "60\u201365%",
    "65%+",
]
over_decided["p_over_bucket"] = pd.cut(
    over_decided["p_over"],
    bins=over_edges,
    labels=over_labels,
    right=False,
    include_lowest=True,
)
over_calibration = (
    over_decided.groupby("p_over_bucket", observed=False)
    .agg(
        n=("over_hit", "size"),
        actual_over_rate=("over_hit", "mean"),
        predicted_p_over=("p_over", "mean"),
    )
    .reset_index()
)
over_calibration["calibration_gap"] = (
    over_calibration["actual_over_rate"]
    - over_calibration["predicted_p_over"]
)
print(
    "Over-side calibration: buckets of model P(over) vs how often the over hit. "
    "Pushes excluded."
)
display(over_calibration.round(3))

ev_edges = [-np.inf, 0.0, 0.05, 0.10, 0.20, np.inf]
ev_labels = ["<0%", "0–5%", "5–10%", "10–20%", "20%+"]
graded["ev_bucket"] = pd.cut(
    graded["ev"], bins=ev_edges, labels=ev_labels, right=False
)
ev_accuracy = (
    graded.groupby("ev_bucket", observed=False)
    .agg(
        n=("result", "size"),
        wins=("result", lambda series: int(series.eq("win").sum())),
        hit_rate=("result", lambda series: series.eq("win").mean()),
        mean_ev=("ev", "mean"),
        mean_probability=("p_side", "mean"),
        roi=("profit", "mean"),
    )
    .reset_index()
)
print("Favored-side results by expected value per $1 stake.")
display(ev_accuracy)

top = (
    book.loc[book["ev"].gt(0)]
    .sort_values("ev", ascending=False)
    .loc[
        :,
        [
            "slate_date",
            "book",
            "player_name",
            "side",
            "line",
            "price",
            "mu",
            "p_side",
            "ev",
            "pts_played",
            "result",
        ],
    ]
    .head(20)
)
print("Highest expected value per $1 stake, at each book's closing price.")
display(top)


Loaded 7,890 point sides on 2026-10-30, 2026-10-31 across 5 books
Main points closes: 1,112. One-sided offers left out: 3,327. Unmatched names (4): Alexandre Sarr, Cam Johnson, Herb Jones, Lu Dort
Scored 1,026 lines with a model mean. 86 had no mean. Alpha 0.3742.
Settled 994. Pushes 4. +EV 884 (853 settled).
Favored side hit 59.0% on 990 decided lines (mean probability 60.9%).
+EV hit 59.6% on 850 decided bets. ROI 12.9% per $1 stake.
Settled closes by book. ROI is profit per $1 at that book's price.


,book,n,decided,hit_rate,roi
0,novig,214.0,214.0,0.570093,0.092403
1,pinnacle,184.0,184.0,0.614130,0.141013
2,prizepicks,216.0,212.0,0.599057,0.100568
3,prophetx,207.0,207.0,0.574879,0.105696
4,underdog,173.0,173.0,0.595376,0.132736


Favored-side calibration by predicted win probability. predicted_prob is the bucket's mean model probability; actual_hit_rate is how often that side won; calibration_gap = actual - predicted (near 0 is well calibrated).


,bucket,n,wins,actual_hit_rate,predicted_prob,roi,calibration_gap
0,50–55%,192,119,0.620,0.529,0.176,0.091
1,55–60%,247,153,0.619,0.575,0.175,0.044
2,60–65%,226,120,0.531,0.626,0.003,-0.095
3,65–70%,162,107,0.660,0.675,0.233,-0.014
4,70%+,115,66,0.574,0.743,0.062,-0.169


Over-side calibration: buckets of model P(over) vs how often the over hit. Pushes excluded.


,p_over_bucket,n,actual_over_rate,predicted_p_over,calibration_gap
0,<40%,500,0.418,0.330,0.088
1,40–45%,210,0.348,0.423,-0.075
2,45–50%,166,0.331,0.472,-0.141
3,50–55%,66,0.485,0.520,-0.035
4,55–60%,41,0.512,0.567,-0.055
5,60–65%,7,0.571,0.615,-0.044
6,65%+,0,NaN,NaN,NaN


Favored-side results by expected value per $1 stake.


,ev_bucket,n,wins,hit_rate,mean_ev,mean_probability,roi
0,<0%,140,77,0.550000,-0.036201,0.519792,0.014493
1,0–5%,124,74,0.596774,0.024631,0.538855,0.120870
2,5–10%,134,85,0.634328,0.076658,0.574853,0.186127
3,10–20%,247,143,0.578947,0.146094,0.610490,0.093008
4,20%+,345,205,0.594203,0.309632,0.683288,0.137260


Highest expected value per $1 stake, at each book's closing price.


,slate_date,book,player_name,side,line,price,mu,p_side,ev,pts_played,result
132,2026-10-31,prizepicks,Bennett Stirtz,under,7.5,-119,3.730396,0.890559,0.638928,11.0,loss
850,2026-10-31,underdog,Bogdan Bogdanovic,under,7.5,102,5.000231,0.785372,0.586452,4.0,win
966,2026-10-31,novig,Bennett Stirtz,under,7.5,-133,3.730396,0.890559,0.560153,11.0,loss
1019,2026-10-31,pinnacle,Jaylen Wells,under,12.5,128,10.481243,0.681043,0.552777,4.0,win
844,2026-10-30,underdog,Terrence Shannon Jr.,under,13.5,102,9.592787,0.767744,0.550844,11.0,win
890,2026-10-31,prophetx,Bennett Stirtz,under,7.5,-136,3.730396,0.890559,0.545382,11.0,loss
415,2026-10-31,novig,Jaylin Williams,under,6.5,-106,4.249588,0.789902,0.535093,3.0,win
667,2026-10-30,novig,Immanuel Quickley,under,17.5,106,13.290889,0.737338,0.518916,17.0,win
532,2026-10-30,prophetx,Scottie Barnes,under,21.5,105,16.332293,0.739191,0.515341,41.0,loss
775,2026-10-30,prophetx,Anfernee Simons,under,10.5,105,7.925747,0.734115,0.504935,21.0,loss


In [9]:
line_edges = [-np.inf, 10, 15, 20, 25, np.inf]
line_labels = ["<10", "10–15", "15–20", "20–25", "25+"]
by_line = graded.copy()
by_line["line_bucket"] = pd.cut(
    by_line["line"], bins=line_edges, labels=line_labels, right=False
)
line_accuracy = (
    by_line.groupby("line_bucket", observed=False)
    .agg(
        n=("result", "size"),
        wins=("result", lambda series: int(series.eq("win").sum())),
        hit_rate=("result", lambda series: series.eq("win").mean()),
        mean_line=("line", "mean"),
        mean_probability=("p_side", "mean"),
        mean_ev=("ev", "mean"),
        roi=("profit", "mean"),
    )
    .reset_index()
)
print(
    "Favored side on decided closes, by the posted points line. "
    "A line of 10 is in 10–15, and 25 is in 25+."
)
display(line_accuracy)


Favored side on decided closes, by the posted points line. A line of 10 is in 10–15, and 25 is in 25+.


,line_bucket,n,wins,hit_rate,mean_line,mean_probability,mean_ev,roi
0,<10,316,177,0.560127,7.643987,0.607425,0.144299,0.049931
1,10–15,315,186,0.590476,12.473016,0.615181,0.162813,0.113214
2,15–20,205,113,0.551220,17.263415,0.601486,0.146918,0.048092
3,20–25,115,81,0.704348,22.639130,0.605070,0.144420,0.331281
4,25+,39,27,0.692308,27.576923,0.628940,0.193727,0.330490
